# End-to-End Credit Risk Analytics & Portfolio Optimization

## Sprint 2 – Data Cleaning & Preprocessing

### Objective

Prepare the raw dataset for exploratory data analysis and predictive modeling by handling missing values, duplicates, incorrect data types, and creating useful features.

In [1]:
import pandas as pd
import numpy as np

# Display all columns
pd.set_option("display.max_columns", None)

# Load dataset
df = pd.read_csv("../data/raw/application_train.csv")

# Create a working copy
df_clean = df.copy()

In [2]:
missing = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Percentage": (df_clean.isnull().sum() / len(df_clean)) * 100
})

missing = missing.sort_values(by="Percentage", ascending=False)

missing.head(20)

,Missing Values,Percentage
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_AVG,214865,69.872297
COMMONAREA_MODE,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_MODE,210199,68.354953
LIVINGAPARTMENTS_AVG,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953


In [3]:
missing_summary = {
    "0% Missing": (missing["Percentage"] == 0).sum(),
    "0-20% Missing": ((missing["Percentage"] > 0) & (missing["Percentage"] <= 20)).sum(),
    "20-40% Missing": ((missing["Percentage"] > 20) & (missing["Percentage"] <= 40)).sum(),
    "40-60% Missing": ((missing["Percentage"] > 40) & (missing["Percentage"] <= 60)).sum(),
    "60%+ Missing": (missing["Percentage"] > 60).sum()
}

pd.DataFrame(
    missing_summary.items(),
    columns=["Missing Range", "Number of Columns"]
)

,Missing Range,Number of Columns
0,0% Missing,55
1,0-20% Missing,17
2,20-40% Missing,1
3,40-60% Missing,32
4,60%+ Missing,17


In [4]:
high_missing = missing[missing["Percentage"] > 60]

high_missing

,Missing Values,Percentage
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_AVG,214865,69.872297
COMMONAREA_MODE,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_MODE,210199,68.354953
LIVINGAPARTMENTS_AVG,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953


In [5]:
high_missing.index.tolist()

['COMMONAREA_MEDI',
 'COMMONAREA_AVG',
 'COMMONAREA_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAPARTMENTS_MEDI',
 'FONDKAPREMONT_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAPARTMENTS_MEDI',
 'FLOORSMIN_AVG',
 'FLOORSMIN_MODE',
 'FLOORSMIN_MEDI',
 'YEARS_BUILD_MEDI',
 'YEARS_BUILD_MODE',
 'YEARS_BUILD_AVG',
 'OWN_CAR_AGE']

## Removing Highly Sparse Columns

Columns with more than 60% missing values were reviewed individually. Columns that were not considered essential for our initial credit risk analysis were removed, while `OWN_CAR_AGE` was retained because its missing values have business meaning.

In [6]:
# Columns to drop
columns_to_drop = [
    "COMMONAREA_MEDI",
    "COMMONAREA_AVG",
    "COMMONAREA_MODE",
    "NONLIVINGAPARTMENTS_MODE",
    "NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAPARTMENTS_MEDI",
    "FONDKAPREMONT_MODE",
    "LIVINGAPARTMENTS_MODE",
    "LIVINGAPARTMENTS_AVG",
    "LIVINGAPARTMENTS_MEDI",
    "FLOORSMIN_AVG",
    "FLOORSMIN_MODE",
    "FLOORSMIN_MEDI",
    "YEARS_BUILD_MEDI",
    "YEARS_BUILD_MODE",
    "YEARS_BUILD_AVG"
]

# Drop the columns
df_clean = df_clean.drop(columns=columns_to_drop)

print("New Shape:", df_clean.shape)

New Shape: (307511, 106)


In [7]:
numerical_missing = df_clean.select_dtypes(include=["int64", "float64"]).columns

num_missing = [
    col for col in numerical_missing
    if df_clean[col].isnull().sum() > 0
]

print("Number of Numerical Columns with Missing Values:", len(num_missing))
num_missing[:20]

Number of Numerical Columns with Missing Values: 46


['AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'OWN_CAR_AGE',
 'CNT_FAM_MEMBERS',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'LANDAREA_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAREA_AVG',
 'APARTMENTS_MODE',
 'BASEMENTAREA_MODE',
 'YEARS_BEGINEXPLUATATION_MODE',
 'ELEVATORS_MODE']

In [8]:
categorical_missing = df_clean.select_dtypes(include=["object"]).columns

cat_missing = [
    col for col in categorical_missing
    if df_clean[col].isnull().sum() > 0
]

print("Number of Categorical Columns with Missing Values:", len(cat_missing))
cat_missing

Number of Categorical Columns with Missing Values: 5


/var/folders/2y/2wtbv1154yz0_9d6fhfm37lw0000gn/T/ipykernel_35242/87246672.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_missing = df_clean.select_dtypes(include=["object"]).columns


['NAME_TYPE_SUITE',
 'OCCUPATION_TYPE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

# Missing Value Treatment Strategy

The missing values were handled based on the nature of each feature.

- Numerical features were imputed using the median to reduce the effect of outliers.
- Categorical features were filled with "Unknown" to preserve missing information.
- OWN_CAR_AGE was excluded from automatic imputation because missing values indicate that an applicant does not own a car.

In [9]:
# Numerical columns with missing values
num_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns

num_cols = [
    col for col in num_cols
    if df_clean[col].isnull().sum() > 0
    and col != "OWN_CAR_AGE"
]

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Numerical missing values handled.")

Numerical missing values handled.


In [10]:
# Categorical columns with missing values
cat_cols = df_clean.select_dtypes(include=["object", "string"]).columns

cat_cols = [
    col for col in cat_cols
    if df_clean[col].isnull().sum() > 0
]

for col in cat_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

print("Categorical missing values handled.")

Categorical missing values handled.


In [11]:
remaining_missing = df_clean.isnull().sum()

remaining_missing = remaining_missing[remaining_missing > 0]

remaining_missing

OWN_CAR_AGE    202929
dtype: int64

## Feature Engineering

Create new business-friendly features that improve interpretability and may enhance model performance.

In [12]:
df_clean["AGE_YEARS"] = (-df_clean["DAYS_BIRTH"] / 365).astype(int)

df_clean[["DAYS_BIRTH", "AGE_YEARS"]].head()

,DAYS_BIRTH,AGE_YEARS
0,-9461,25
1,-16765,45
2,-19046,52
3,-19005,52
4,-19932,54


In [13]:
df_clean["YEARS_EMPLOYED"] = abs(df_clean["DAYS_EMPLOYED"]) / 365

df_clean[["DAYS_EMPLOYED", "YEARS_EMPLOYED"]].head()

,DAYS_EMPLOYED,YEARS_EMPLOYED
0,-637,1.745205
1,-1188,3.254795
2,-225,0.616438
3,-3039,8.326027
4,-3038,8.323288


In [14]:
df_clean["CREDIT_INCOME_RATIO"] = (
    df_clean["AMT_CREDIT"] / df_clean["AMT_INCOME_TOTAL"]
)

df_clean[["AMT_CREDIT", "AMT_INCOME_TOTAL", "CREDIT_INCOME_RATIO"]].head()

,AMT_CREDIT,AMT_INCOME_TOTAL,CREDIT_INCOME_RATIO
0,406597.5,202500.0,2.007889
1,1293502.5,270000.0,4.790750
2,135000.0,67500.0,2.000000
3,312682.5,135000.0,2.316167
4,513000.0,121500.0,4.222222


In [15]:
df_clean["ANNUITY_INCOME_RATIO"] = (
    df_clean["AMT_ANNUITY"] / df_clean["AMT_INCOME_TOTAL"]
)

df_clean[["AMT_ANNUITY", "AMT_INCOME_TOTAL", "ANNUITY_INCOME_RATIO"]].head()

,AMT_ANNUITY,AMT_INCOME_TOTAL,ANNUITY_INCOME_RATIO
0,24700.5,202500.0,0.121978
1,35698.5,270000.0,0.132217
2,6750.0,67500.0,0.100000
3,29686.5,135000.0,0.219900
4,21865.5,121500.0,0.179963


In [16]:
df_clean["HAS_CAR"] = (
    df_clean["FLAG_OWN_CAR"].map({"Y": 1, "N": 0})
)

df_clean[["FLAG_OWN_CAR", "HAS_CAR"]].head()

,FLAG_OWN_CAR,HAS_CAR
0,N,0
1,N,0
2,Y,1
3,N,0
4,N,0


In [17]:
df_clean["HAS_REALTY"] = (
    df_clean["FLAG_OWN_REALTY"].map({"Y": 1, "N": 0})
)

df_clean[["FLAG_OWN_REALTY", "HAS_REALTY"]].head()

,FLAG_OWN_REALTY,HAS_REALTY
0,Y,1
1,N,0
2,Y,1
3,Y,1
4,Y,1


In [18]:
df_clean.shape

(307511, 112)

In [19]:
df_clean[
    [
        "AGE_YEARS",
        "YEARS_EMPLOYED",
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "HAS_CAR",
        "HAS_REALTY",
    ]
].head()

,AGE_YEARS,YEARS_EMPLOYED,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,HAS_CAR,HAS_REALTY
0,25,1.745205,2.007889,0.121978,0,1
1,45,3.254795,4.790750,0.132217,0,0
2,52,0.616438,2.000000,0.100000,1,1
3,52,8.326027,2.316167,0.219900,0,1
4,54,8.323288,4.222222,0.179963,0,1


In [20]:
(df_clean["DAYS_EMPLOYED"] == 365243).sum()

np.int64(55374)

In [21]:
df_clean["DAYS_EMPLOYED"].describe()

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

## Handling Special Placeholder Values

The `DAYS_EMPLOYED` column contains a placeholder value (`365243`) used to represent unknown employment information. These values were replaced with missing values and then imputed using the median before creating the employment duration feature.

In [22]:
# Count placeholder values
placeholder_count = (df_clean["DAYS_EMPLOYED"] == 365243).sum()

print("Placeholder values:", placeholder_count)

Placeholder values: 55374


In [23]:
# Replace placeholder with NaN
df_clean["DAYS_EMPLOYED"] = df_clean["DAYS_EMPLOYED"].replace(365243, np.nan)

print("Replacement completed.")

Replacement completed.


In [24]:
# Fill missing values with median
median_days = df_clean["DAYS_EMPLOYED"].median()

df_clean["DAYS_EMPLOYED"] = df_clean["DAYS_EMPLOYED"].fillna(median_days)

print("Median used:", median_days)

Median used: -1648.0


In [25]:
# Recalculate YEARS_EMPLOYED
df_clean["YEARS_EMPLOYED"] = abs(df_clean["DAYS_EMPLOYED"]) / 365

df_clean[["DAYS_EMPLOYED", "YEARS_EMPLOYED"]].head()

,DAYS_EMPLOYED,YEARS_EMPLOYED
0,-637.0,1.745205
1,-1188.0,3.254795
2,-225.0,0.616438
3,-3039.0,8.326027
4,-3038.0,8.323288


In [26]:
(df_clean["DAYS_EMPLOYED"] == 365243).sum()

np.int64(0)

In [27]:
# Check duplicate rows
duplicates = df_clean.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [28]:
df_clean.dtypes.value_counts()

float64    54
int64      43
str        15
Name: count, dtype: int64

In [29]:
df_clean.isnull().sum().sort_values(ascending=False).head(10)

OWN_CAR_AGE                 202929
SK_ID_CURR                       0
FLAG_DOCUMENT_6                  0
FLAG_DOCUMENT_4                  0
FLAG_DOCUMENT_3                  0
FLAG_DOCUMENT_2                  0
DAYS_LAST_PHONE_CHANGE           0
DEF_60_CNT_SOCIAL_CIRCLE         0
OBS_60_CNT_SOCIAL_CIRCLE         0
DEF_30_CNT_SOCIAL_CIRCLE         0
dtype: int64

In [30]:
import os

# Create processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save cleaned dataset
df_clean.to_csv("../data/processed/application_train_clean.csv", index=False)

print("✅ Cleaned dataset saved successfully!")

✅ Cleaned dataset saved successfully!
